# Colab runner

Runs training and evaluation on Colab's GPU, using the code in the GitHub repository, and pushes the results back.
The notebook holds no logic of its own. All code lives in `src/ssp/`.

**Before the first run**
1. The repository is on GitHub (see the README, "Connect GitHub").
2. On GitHub, create a fine-grained personal access token limited to this repository, with **Contents: Read and write**.
3. In Colab, open the key icon in the left bar (Secrets), add a secret named `GH_TOKEN` holding that token, and allow this notebook to read it.
4. Runtime, Change runtime type, choose a GPU.

Each session: run the cells from top to bottom.

In [ ]:
# 1. Settings: change these
GITHUB_USER = "your-github-username"
REPO = "scanner-shift-proxy"
YOUR_NAME = "Your Name"            # shown on the commits this notebook makes
YOUR_EMAIL = "you@example.com"
MODELS = ["m01"]                   # the models to train and evaluate in this session

In [ ]:
# 2. Get the code and install it
import os, subprocess
from google.colab import userdata

token = userdata.get("GH_TOKEN")
workdir = f"/content/{REPO}"
if not os.path.exists(workdir):
    subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git", workdir], check=True)
os.chdir(workdir)
subprocess.run(["git", "pull", "-q", "--rebase"], check=True)
subprocess.run(["git", "config", "user.name", YOUR_NAME], check=True)
subprocess.run(["git", "config", "user.email", YOUR_EMAIL], check=True)
!pip -q install -e ".[train]"
!git log --oneline -1

In [ ]:
# 3. Data and storage
# Model weights go to Google Drive so they survive the session. The dataset is
# fetched from Kaggle onto the Colab machine (nothing is downloaded to your laptop).
from google.colab import drive
drive.mount("/content/drive")
os.environ["SSP_RUNS"] = "/content/drive/MyDrive/ssp_runs"

import kagglehub
os.environ["MNMS_ROOT"] = kagglehub.dataset_download("tailength/m-and-ms-dataset")
print("Data:", os.environ["MNMS_ROOT"])

In [ ]:
# 4. Where the project stands
!python -m ssp.status

In [ ]:
# 5. The split. Made ONCE for the whole project, by one person, then committed (cell 8).
if not os.path.exists("configs/split.json"):
    !python -m ssp.split
else:
    print("Split already exists. Not touching it.")

In [ ]:
# 6. Train
for m in MODELS:
    !python -m ssp.train --model {m}

In [ ]:
# 7. Evaluate
# The clean arm can run at any time. The sim and real arms are refused by the code
# until configs/severity_grid.yaml says 'status: frozen' and that is committed.
for m in MODELS:
    !python -m ssp.evaluate --model {m} --arm clean

# After the freeze, uncomment:
# for m in MODELS:
#     !python -m ssp.evaluate --model {m} --arm sim
#     !python -m ssp.evaluate --model {m} --arm real

In [ ]:
# 8. Send the results back to GitHub
msg = "Results: " + " ".join(MODELS)
!git add results configs/split.json
!git commit -q -m "{msg}" || echo "Nothing new to commit"
!git pull -q --rebase && git push -q
!git log --oneline -3
!python -m ssp.status

## Running on Kaggle Notebooks instead

Attach the dataset with "Add Input", then replace cell 3 with:

```python
os.environ["SSP_RUNS"] = "/kaggle/working/ssp_runs"
os.environ["MNMS_ROOT"] = "/kaggle/input/m-and-ms-dataset"
```

and in cell 2 read the token from Kaggle's Secrets (`from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret("GH_TOKEN")`) and use `/kaggle/working` in place of `/content`.
Weights in `/kaggle/working` are kept only if you save the notebook version, so download `best.pt` files you need.